# 정상-only GNSS 스푸핑 탐지 모델 학습 Notebook

목표는 **스푸핑 데이터를 학습하지 않고**, 정상 GNSS tracking observable만으로 다음 두 가지를 학습하는 것입니다.

1. **PRN별 정상 3-tap tracking morphology**  
   Early/Prompt/Late correlator shape와 Doppler/code loop dynamics가 정상 상태에서 시간적으로 어떻게 변하는지 학습합니다.

2. **정상 상태의 PRN 간 관계**  
   PRN들의 원시 peak 모양이 서로 같아야 한다는 뜻이 아닙니다. PRN마다 elevation, azimuth, Doppler, multipath가 다릅니다.  
   여기서 학습/탐지하려는 관계는 **여러 PRN의 변화, 예측오차, onset이 정상 상태에서 어떤 공통 패턴을 갖는지**입니다.

핵심 탐지 논리:

```text
정상 데이터 only
→ PRN별 3-tap morphology + loop dynamics window 생성
→ 정상 시간 진화 예측 / masked PRN 복원
→ spoofing-like event에서는 여러 PRN의 예측오차·복원오차가 동시/common-mode로 증가
```

이 Notebook은 먼저 PoC가 바로 돌아가도록 설계했습니다.

- 데이터가 있으면 GNSS-SDR tracking dump CSV/Parquet을 읽습니다.
- 데이터가 없으면 작은 synthetic normal sample로 notebook 구조를 검증합니다.
- main detector는 power shortcut을 피하기 위해 `C/N0 mean`, raw prompt magnitude mean, AGC를 기본 feature에서 제외합니다.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    from sklearn.preprocessing import RobustScaler, StandardScaler
    from sklearn.ensemble import IsolationForest
    from sklearn.covariance import EmpiricalCovariance
    SKLEARN_OK = True
except Exception as e:
    SKLEARN_OK = False
    print('scikit-learn unavailable:', e)

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False
    print('PyTorch unavailable:', e)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / 'artifacts'
OUT_DIR = PROJECT_ROOT / 'artifacts' / 'normal_only_prn_graph_training'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUT_DIR      =', OUT_DIR)
print('SKLEARN_OK   =', SKLEARN_OK)
print('TORCH_OK     =', TORCH_OK)

## 1. 설정

`TRACKING_FILES`에 정상 데이터 tracking dump 경로를 넣습니다.

예상 입력은 GNSS-SDR tracking dump에서 나온 CSV/Parquet입니다. 컬럼명은 환경마다 다를 수 있으므로 아래 loader가 가능한 한 자동으로 맞춥니다.

필수적으로 필요한 정보:

```text
time
PRN/channel identifier
E_I, E_Q 또는 abs_E
P_I, P_Q 또는 abs_P
L_I, L_Q 또는 abs_L
```

있으면 쓰는 보조 정보:

```text
carrier_doppler_hz
code_error
prompt I/Q phase
cn0_db_hz: 기본 detector에는 mean을 쓰지 않고 std/slope만 사용
```

In [ ]:
@dataclass
class Config:
    # 정상 tracking dump 파일들. 비워두면 synthetic normal sample을 생성합니다.
    TRACKING_FILES: tuple[str, ...] = tuple()

    # window 설정
    WINDOW_SEC: float = 1.0
    STRIDE_SEC: float = 1.0
    MIN_EPOCHS_PER_PRN_WINDOW: int = 5
    MIN_PRNS_PER_RECEIVER_WINDOW: int = 4

    # sequence 학습 설정
    HISTORY_LEN: int = 5        # 최근 5초로 다음 1초를 예측
    MASK_PROB: float = 0.25     # masked PRN reconstruction용

    # feature 설정
    EPS: float = 1e-6
    USE_CN0_STD_ONLY: bool = True
    INCLUDE_POWER_ABLATION: bool = False  # True로 켜면 prompt magnitude mean 등 ablation feature 추가

    # 모델/학습 설정
    RANDOM_SEED: int = 42
    BATCH_SIZE: int = 64
    EPOCHS: int = 20
    LR: float = 1e-3

CFG = Config(
    TRACKING_FILES=tuple(),  # 예: ('artifacts/receiver/run_x/tracking_dump.csv',)
)

np.random.seed(CFG.RANDOM_SEED)
if TORCH_OK:
    torch.manual_seed(CFG.RANDOM_SEED)

CFG

## 2. Tracking dump 로드

컬럼명은 수신기 설정마다 다릅니다. 아래 함수는 자주 쓰이는 이름을 정규화합니다.

정규화 후 목표 schema:

```text
time_s
prn
E, P, L
carrier_doppler_hz optional
code_error optional
prompt_phase_rad optional
cn0_db_hz optional
```

In [ ]:
def _first_existing(df: pd.DataFrame, candidates: list[str]) -> str | None:
    lower_map = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name in df.columns:
            return name
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    return None


def _mag_from_iq(df: pd.DataFrame, i_candidates: list[str], q_candidates: list[str]) -> pd.Series | None:
    i_col = _first_existing(df, i_candidates)
    q_col = _first_existing(df, q_candidates)
    if i_col and q_col:
        return np.sqrt(df[i_col].astype(float)**2 + df[q_col].astype(float)**2)
    return None


def normalize_tracking_df(df: pd.DataFrame, source: str = '') -> pd.DataFrame:
    out = pd.DataFrame()

    time_col = _first_existing(df, ['time_s', 'time', 'timestamp', 'tracking_time_s', 'tow', 'rx_time', 'sample_time_s'])
    if time_col is None:
        raise ValueError(f'No time column found in {source}. columns={list(df.columns)[:30]}')
    out['time_s'] = df[time_col].astype(float)

    prn_col = _first_existing(df, ['prn', 'PRN', 'satellite', 'sv', 'system_prn', 'channel_prn', 'PRN_id'])
    if prn_col is None:
        chan_col = _first_existing(df, ['channel', 'Channel'])
        if chan_col is None:
            raise ValueError(f'No PRN/channel column found in {source}.')
        out['prn'] = df[chan_col].astype(str)
    else:
        out['prn'] = df[prn_col].astype(str).str.upper().str.replace(' ', '', regex=False)
        # 숫자만 있으면 Gxx로 맞춤
        mask_num = out['prn'].str.fullmatch(r'\d+')
        out.loc[mask_num, 'prn'] = out.loc[mask_num, 'prn'].astype(int).map(lambda x: f'G{x:02d}')

    # E/P/L magnitude. abs_*가 있으면 우선 사용, 없으면 I/Q로 계산.
    e_abs = _first_existing(df, ['abs_E', 'E_abs', 'early_abs', 'Early_abs'])
    p_abs = _first_existing(df, ['abs_P', 'P_abs', 'prompt_abs', 'Prompt_abs'])
    l_abs = _first_existing(df, ['abs_L', 'L_abs', 'late_abs', 'Late_abs'])

    E = df[e_abs].astype(float) if e_abs else _mag_from_iq(df, ['E_I','early_I','Early_I','i_E','E'], ['E_Q','early_Q','Early_Q','q_E'])
    P = df[p_abs].astype(float) if p_abs else _mag_from_iq(df, ['P_I','prompt_I','Prompt_I','i_P','P'], ['P_Q','prompt_Q','Prompt_Q','q_P'])
    L = df[l_abs].astype(float) if l_abs else _mag_from_iq(df, ['L_I','late_I','Late_I','i_L','L'], ['L_Q','late_Q','Late_Q','q_L'])

    if E is None or P is None or L is None:
        raise ValueError(f'Need E/P/L magnitudes or I/Q columns in {source}. columns={list(df.columns)[:50]}')

    out['E'] = E.astype(float)
    out['P'] = P.astype(float)
    out['L'] = L.astype(float)

    # Optional loop dynamics
    opt_map = {
        'carrier_doppler_hz': ['carrier_doppler_hz','doppler_hz','carrier_doppler','Doppler_Hz','doppler'],
        'code_error': ['code_error','dll_error','code_phase_error','tracking_error_chips'],
        'cn0_db_hz': ['cn0_db_hz','CN0_dB_Hz','cno_dbhz','cn0','C/N0'],
    }
    for out_col, candidates in opt_map.items():
        c = _first_existing(df, candidates)
        if c:
            out[out_col] = pd.to_numeric(df[c], errors='coerce')

    # Prompt phase from prompt I/Q when available
    p_i = _first_existing(df, ['P_I','prompt_I','Prompt_I','i_P'])
    p_q = _first_existing(df, ['P_Q','prompt_Q','Prompt_Q','q_P'])
    if p_i and p_q:
        out['prompt_phase_rad'] = np.unwrap(np.arctan2(df[p_q].astype(float), df[p_i].astype(float)))

    out['source'] = source
    out = out.replace([np.inf, -np.inf], np.nan).dropna(subset=['time_s','prn','E','P','L'])
    out = out.sort_values(['time_s','prn']).reset_index(drop=True)
    return out


def load_tracking_files(paths: tuple[str, ...]) -> pd.DataFrame:
    frames = []
    for p in paths:
        path = Path(p)
        if not path.is_absolute():
            path = PROJECT_ROOT / path
        if not path.exists():
            warnings.warn(f'Missing file: {path}')
            continue
        if path.suffix.lower() in ['.parquet', '.pq']:
            raw = pd.read_parquet(path)
        else:
            raw = pd.read_csv(path)
        frames.append(normalize_tracking_df(raw, source=str(path)))
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

## 3. 데이터가 없을 때 notebook 검증용 synthetic normal 생성

실제 학습에는 이 synthetic 데이터를 쓰지 않습니다.  
Notebook 구조와 feature/model 코드가 깨지지 않는지 확인하기 위한 더미 정상 데이터입니다.

In [ ]:
def make_synthetic_normal_tracking(n_seconds=300, hz=20, prns=('G03','G08','G14','G17','G22','G27','G31')):
    rng = np.random.default_rng(CFG.RANDOM_SEED)
    times = np.arange(0, n_seconds, 1/hz)
    rows = []
    for prn_idx, prn in enumerate(prns):
        elev_factor = 0.8 + 0.25*np.sin(0.003*times + prn_idx)
        multipath = 0.02*np.sin(0.09*times + prn_idx*0.7)
        P = 1.0 * elev_factor + 0.02*rng.normal(size=len(times))
        asym = multipath + 0.01*rng.normal(size=len(times))
        E = 0.62*P*(1 + asym) + 0.01*rng.normal(size=len(times))
        L = 0.62*P*(1 - asym) + 0.01*rng.normal(size=len(times))
        dop = -600 + prn_idx*120 + 15*np.sin(0.01*times + prn_idx) + rng.normal(scale=0.5, size=len(times))
        code_err = 0.02*np.sin(0.05*times + prn_idx) + 0.01*rng.normal(size=len(times))
        cn0 = 43 + 2*elev_factor + rng.normal(scale=0.4, size=len(times))
        phase = np.unwrap(0.1*times + 0.05*np.sin(0.05*times) + rng.normal(scale=0.02, size=len(times)))
        rows.extend(dict(time_s=t, prn=prn, E=e, P=p, L=l, carrier_doppler_hz=d, code_error=c, cn0_db_hz=n, prompt_phase_rad=ph, source='synthetic_normal')
                    for t,e,p,l,d,c,n,ph in zip(times,E,P,L,dop,code_err,cn0,phase))
    return pd.DataFrame(rows)

tracking = load_tracking_files(CFG.TRACKING_FILES)
if tracking.empty:
    print('No tracking files configured/found. Using synthetic normal data for notebook smoke test.')
    tracking = make_synthetic_normal_tracking()

print(tracking.shape)
tracking.head()

## 4. 3-tap morphology feature 생성

여기서의 핵심은 `E/P/L`의 **절대 세기**가 아니라 normalized shape입니다.

```text
near_sym         = (E - L) / (E + L + eps)
sharpness        = (2P - E - L) / (P + eps)
prompt_dominance = P / (E + L + eps)
peak_com_3tap    = (-E + L) / (E + P + L + eps)
```

이 feature들이 정상 tracking peak morphology의 기본 표현입니다.

In [ ]:
def add_epoch_features(df: pd.DataFrame, eps: float = 1e-6) -> pd.DataFrame:
    x = df.copy()
    E, P, L = x['E'].astype(float), x['P'].astype(float), x['L'].astype(float)
    x['near_sym'] = (E - L) / (E + L + eps)
    x['sharpness'] = (2*P - E - L) / (P + eps)
    x['prompt_dominance'] = P / (E + L + eps)
    x['peak_com_3tap'] = (-E + L) / (E + P + L + eps)
    s = E + P + L + eps
    e, p, l = E/s, P/s, L/s
    x['shape_entropy'] = -(e*np.log(e+eps) + p*np.log(p+eps) + l*np.log(l+eps))
    return x

epoch = add_epoch_features(tracking, CFG.EPS)
epoch[['time_s','prn','near_sym','sharpness','prompt_dominance','peak_com_3tap','shape_entropy']].head()

## 5. PRN별 1초 window feature 생성

각 PRN별로 window 안에서 morphology와 loop dynamics의 `mean/std/slope/delta`를 계산합니다.

논문 메인 feature는 아래를 중심으로 둡니다.

- morphology: `near_sym`, `sharpness`, `prompt_dominance`, `peak_com_3tap`, `shape_entropy`
- loop dynamics: `carrier_doppler_hz`, `code_error`, `prompt_phase_rad`
- C/N0는 기본적으로 `std/slope/delta`만 사용하고 `mean`은 power shortcut 우려 때문에 제외합니다.

In [ ]:
def slope_of(y: np.ndarray, t: np.ndarray) -> float:
    y = np.asarray(y, dtype=float)
    t = np.asarray(t, dtype=float)
    ok = np.isfinite(y) & np.isfinite(t)
    if ok.sum() < 2:
        return np.nan
    tt = t[ok] - t[ok].mean()
    yy = y[ok] - y[ok].mean()
    denom = np.dot(tt, tt)
    if denom <= 0:
        return 0.0
    return float(np.dot(tt, yy) / denom)

BASE_EPOCH_FEATURES = ['near_sym','sharpness','prompt_dominance','peak_com_3tap','shape_entropy']
OPTIONAL_DYNAMICS = [c for c in ['carrier_doppler_hz','code_error','prompt_phase_rad','cn0_db_hz'] if c in epoch.columns]
print('Optional dynamics:', OPTIONAL_DYNAMICS)


def make_prn_windows(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    t_min, t_max = float(df['time_s'].min()), float(df['time_s'].max())
    starts = np.arange(t_min, t_max - cfg.WINDOW_SEC + 1e-9, cfg.STRIDE_SEC)
    rows = []
    feature_cols = BASE_EPOCH_FEATURES + OPTIONAL_DYNAMICS
    for start in starts:
        end = start + cfg.WINDOW_SEC
        w = df[(df['time_s'] >= start) & (df['time_s'] < end)]
        if w.empty:
            continue
        for prn, g in w.groupby('prn'):
            if len(g) < cfg.MIN_EPOCHS_PER_PRN_WINDOW:
                continue
            row = {'window_start_s': start, 'window_end_s': end, 'window_center_s': start + cfg.WINDOW_SEC/2, 'prn': prn, 'n_epochs': len(g)}
            tt = g['time_s'].to_numpy()
            for col in feature_cols:
                vals = pd.to_numeric(g[col], errors='coerce').to_numpy(dtype=float)
                if np.isfinite(vals).sum() == 0:
                    continue
                # cn0 mean은 기본 detector에서 제외
                include_mean = not (col == 'cn0_db_hz' and cfg.USE_CN0_STD_ONLY)
                if include_mean:
                    row[f'{col}_mean'] = float(np.nanmean(vals))
                row[f'{col}_std'] = float(np.nanstd(vals))
                row[f'{col}_slope'] = slope_of(vals, tt)
                row[f'{col}_delta'] = float(vals[-1] - vals[0]) if len(vals) >= 2 else np.nan
            if cfg.INCLUDE_POWER_ABLATION:
                # ablation 전용: main detector feature가 아님
                row['prompt_mag_mean_ablation'] = float(np.nanmean(g['P']))
                row['prompt_mag_std_ablation'] = float(np.nanstd(g['P']))
            rows.append(row)
    out = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan)
    return out

prn_windows = make_prn_windows(epoch, CFG)
print(prn_windows.shape)
prn_windows.head()

## 6. Receiver-level graph window 구성

한 시간창의 전체 PRN 집합을 하나의 graph sample로 봅니다.

```text
node = tracked PRN
node feature = 해당 PRN의 1초 window feature
edge = 초기 PoC에서는 fully-connected
```

중요: 여기서 말하는 PRN 관계 유사성은 `G03 peak shape == G08 peak shape`가 아닙니다.  
정상 상태에서 **PRN들의 변화량, 예측오차, onset이 얼마나 공통적으로 움직이는지**를 보는 것입니다.

In [ ]:
NON_FEATURE_COLS = {'window_start_s','window_end_s','window_center_s','prn','n_epochs'}
feature_cols = [c for c in prn_windows.columns if c not in NON_FEATURE_COLS]
# 결측이 너무 많은 컬럼 제거
na_rate = prn_windows[feature_cols].isna().mean().sort_values(ascending=False)
feature_cols = [c for c in feature_cols if na_rate[c] < 0.2]
prn_windows = prn_windows.dropna(subset=feature_cols).reset_index(drop=True)

print('n window rows:', len(prn_windows))
print('n feature cols:', len(feature_cols))
print(feature_cols[:20])

# receiver window별 최소 PRN 수 필터
counts = prn_windows.groupby('window_center_s')['prn'].nunique()
valid_centers = counts[counts >= CFG.MIN_PRNS_PER_RECEIVER_WINDOW].index
prn_windows = prn_windows[prn_windows['window_center_s'].isin(valid_centers)].reset_index(drop=True)
print('after min PRNs filter:', prn_windows.shape)
print('receiver windows:', prn_windows['window_center_s'].nunique())

## 7. 정상 데이터 scaling

정상-only 학습에서는 train normal 통계로만 scaling합니다.

실제 연구에서는 run/day/location/trajectory 단위로 train/validation을 나눠야 합니다. 이 Notebook PoC는 시간 순서 기준으로 앞 70%를 train, 뒤 30%를 validation으로 둡니다.

In [ ]:
centers = np.array(sorted(prn_windows['window_center_s'].unique()))
split_idx = int(len(centers) * 0.7)
train_centers = set(centers[:split_idx])
val_centers = set(centers[split_idx:])

train_df = prn_windows[prn_windows['window_center_s'].isin(train_centers)].copy()
val_df = prn_windows[prn_windows['window_center_s'].isin(val_centers)].copy()

scaler = RobustScaler() if SKLEARN_OK else None
if scaler is not None:
    scaler.fit(train_df[feature_cols].to_numpy())
    train_df.loc[:, feature_cols] = scaler.transform(train_df[feature_cols].to_numpy())
    val_df.loc[:, feature_cols] = scaler.transform(val_df[feature_cols].to_numpy())

print('train:', train_df.shape, 'val:', val_df.shape)
train_df.head()

## 8. Baseline 1 — PRN별 정상 분포 학습

먼저 단순 baseline으로 per-PRN window feature의 정상 분포를 학습합니다.

- Isolation Forest
- Mahalanobis distance

이 baseline은 PRN 간 관계를 직접 학습하지 않지만, 이후 receiver-level aggregation에서 여러 PRN의 score 관계를 볼 수 있습니다.

In [ ]:
def topk_mean(arr, k=3):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan
    k = min(k, len(arr))
    return float(np.sort(arr)[-k:].mean())

if SKLEARN_OK and len(train_df) > 10:
    X_train = train_df[feature_cols].to_numpy(dtype=float)
    X_val = val_df[feature_cols].to_numpy(dtype=float)

    iso = IsolationForest(n_estimators=200, contamination='auto', random_state=CFG.RANDOM_SEED)
    iso.fit(X_train)
    train_df['score_iso'] = -iso.score_samples(X_train)
    val_df['score_iso'] = -iso.score_samples(X_val)

    cov = EmpiricalCovariance().fit(X_train)
    train_df['score_maha'] = cov.mahalanobis(X_train)
    val_df['score_maha'] = cov.mahalanobis(X_val)

    score_col = 'score_iso'
    thr = np.quantile(train_df[score_col], 0.995)
    print('threshold 99.5% train:', thr)

    def aggregate_scores(df, score_col):
        rows = []
        for t, g in df.groupby('window_center_s'):
            s = g[score_col].to_numpy(dtype=float)
            rows.append({
                'window_center_s': t,
                'tracked_prn_count': g['prn'].nunique(),
                'score_median': float(np.nanmedian(s)),
                'score_max': float(np.nanmax(s)),
                'score_top3_mean': topk_mean(s, 3),
                'frac_prn_above_thr': float(np.mean(s > thr)),
            })
        return pd.DataFrame(rows)

    receiver_train = aggregate_scores(train_df, score_col)
    receiver_val = aggregate_scores(val_df, score_col)
    display(receiver_train.head())
    display(receiver_val.head())
else:
    print('Skipping sklearn baselines.')

## 9. PRN 간 관계 점수 — common-mode error

PRN 간 관계는 raw feature similarity가 아니라 **동시적/공통적 변화**로 정의합니다.

한 receiver window에서 PRN별 feature 또는 anomaly error matrix를 만들고 PCA/SVD로 공통 모드 에너지를 계산합니다.

```text
common_mode_energy = PC1 explained variance ratio
common_mode_score  = common_mode_energy × mean_error_magnitude
```

이렇게 해야 단순히 모든 PRN이 정상적으로 함께 조금 움직인 경우와, 실제 anomaly magnitude가 커진 경우를 구분할 수 있습니다.

In [ ]:
def pc1_explained_ratio(X: np.ndarray) -> float:
    X = np.asarray(X, dtype=float)
    X = X[np.isfinite(X).all(axis=1)]
    if X.shape[0] < 2 or X.shape[1] < 2:
        return np.nan
    X = X - X.mean(axis=0, keepdims=True)
    norm = np.linalg.norm(X)
    if norm <= 0:
        return 0.0
    _, s, _ = np.linalg.svd(X, full_matrices=False)
    var = s**2
    return float(var[0] / (var.sum() + 1e-12))


def add_common_mode_from_feature_deltas(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    # 정상 관계의 변화 유사성: 각 PRN의 이전 window 대비 delta feature matrix로 계산
    d = df.sort_values(['prn','window_center_s']).copy()
    for c in cols:
        d[f'delta_{c}'] = d.groupby('prn')[c].diff()
    delta_cols = [f'delta_{c}' for c in cols]
    rows = []
    for t, g in d.groupby('window_center_s'):
        X = g[delta_cols].to_numpy(dtype=float)
        rows.append({
            'window_center_s': t,
            'common_mode_feature_delta': pc1_explained_ratio(X),
            'delta_magnitude_mean': float(np.nanmean(np.linalg.norm(np.nan_to_num(X), axis=1))) if len(g) else np.nan,
        })
    out = pd.DataFrame(rows)
    out['common_mode_delta_score'] = out['common_mode_feature_delta'] * out['delta_magnitude_mean']
    return out

core_delta_cols = [c for c in feature_cols if any(k in c for k in ['near_sym','sharpness','prompt_dominance','peak_com_3tap','carrier_doppler_hz','code_error'])]
core_delta_cols = core_delta_cols[:20]  # PoC: 너무 많은 컬럼을 피함
common_train = add_common_mode_from_feature_deltas(train_df, core_delta_cols)
common_val = add_common_mode_from_feature_deltas(val_df, core_delta_cols)
common_val.head()

## 10. Main model PoC — 정상 sequence next-window prediction

아래는 정상 데이터만으로 학습하는 sequence predictor입니다.

간단화를 위해 PoC v1에서는 PRN별 sequence를 독립적으로 예측합니다.

```text
입력: PRN i의 최근 K개 window feature
목표: 같은 PRN i의 다음 window feature
score_i(t+1) = prediction error
```

그 다음 receiver-level에서 여러 PRN의 prediction error가 동시에 커지는지를 봅니다.

> v2에서는 여기에 masked-PRN reconstruction을 추가하고, v3에서는 Graph Attention/geometry-aware edge를 넣습니다.

In [ ]:
if TORCH_OK:
    class PRNSequenceDataset(Dataset):
        def __init__(self, df: pd.DataFrame, feature_cols: list[str], history_len: int):
            self.X = []
            self.y = []
            self.meta = []
            for prn, g in df.sort_values('window_center_s').groupby('prn'):
                vals = g[feature_cols].to_numpy(dtype=np.float32)
                times = g['window_center_s'].to_numpy(dtype=float)
                for i in range(history_len, len(g)):
                    self.X.append(vals[i-history_len:i])
                    self.y.append(vals[i])
                    self.meta.append({'prn': prn, 'window_center_s': times[i]})
            self.X = torch.tensor(np.stack(self.X), dtype=torch.float32) if self.X else torch.empty(0, history_len, len(feature_cols))
            self.y = torch.tensor(np.stack(self.y), dtype=torch.float32) if self.y else torch.empty(0, len(feature_cols))
        def __len__(self): return len(self.X)
        def __getitem__(self, idx): return self.X[idx], self.y[idx]

    class GRUPredictor(nn.Module):
        def __init__(self, n_features: int, hidden: int = 96):
            super().__init__()
            self.gru = nn.GRU(input_size=n_features, hidden_size=hidden, batch_first=True)
            self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, n_features))
        def forward(self, x):
            h, _ = self.gru(x)
            return self.head(h[:, -1, :])

    train_ds = PRNSequenceDataset(train_df, feature_cols, CFG.HISTORY_LEN)
    val_ds = PRNSequenceDataset(val_df, feature_cols, CFG.HISTORY_LEN)
    print('train sequences:', len(train_ds), 'val sequences:', len(val_ds))
else:
    print('PyTorch unavailable; skip sequence model.')

In [ ]:
if TORCH_OK and 'train_ds' in globals() and len(train_ds) > 10:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = GRUPredictor(n_features=len(feature_cols)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.LR)
    loss_fn = nn.SmoothL1Loss()
    loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True)

    history = []
    for epoch_idx in range(1, CFG.EPOCHS + 1):
        model.train()
        losses = []
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(float(loss.detach().cpu()))
        history.append({'epoch': epoch_idx, 'train_loss': float(np.mean(losses))})
        if epoch_idx % max(1, CFG.EPOCHS//5) == 0 or epoch_idx == 1:
            print(history[-1])

    torch.save({'model_state_dict': model.state_dict(), 'feature_cols': feature_cols, 'config': CFG.__dict__}, OUT_DIR / 'gru_next_window_predictor.pt')
    pd.DataFrame(history).to_csv(OUT_DIR / 'training_history.csv', index=False)
else:
    print('Skipping GRU training.')

## 11. Prediction error → PRN score → receiver score

정상 validation에서 threshold를 잡습니다.

TEXBAT 같은 외부 spoofing 데이터에도 같은 절차를 적용합니다.

```text
TEXBAT raw IQ
→ GNSS-SDR tracking dump
→ 같은 feature/window/scaler
→ model prediction error
→ PRN score
→ receiver-level common-mode aggregation
```

In [ ]:
def score_sequence_dataset(model, ds, meta, device='cpu'):
    model.eval()
    scores = []
    with torch.no_grad():
        for idx in range(len(ds)):
            xb, yb = ds[idx]
            pred = model(xb.unsqueeze(0).to(device)).cpu().numpy()[0]
            y = yb.numpy()
            err_vec = np.abs(pred - y)
            rec = dict(meta[idx])
            rec['pred_l1_mean'] = float(err_vec.mean())
            rec['pred_l2'] = float(np.sqrt((err_vec**2).mean()))
            scores.append(rec)
    return pd.DataFrame(scores)

if TORCH_OK and 'model' in globals() and len(val_ds) > 0:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    prn_val_scores = score_sequence_dataset(model, val_ds, val_ds.meta, device=device)
    thr_prn = np.quantile(prn_val_scores['pred_l1_mean'], 0.995)
    print('PRN score threshold from validation 99.5%:', thr_prn)

    receiver_scores = []
    for t, g in prn_val_scores.groupby('window_center_s'):
        s = g['pred_l1_mean'].to_numpy()
        receiver_scores.append({
            'window_center_s': t,
            'tracked_prn_count': g['prn'].nunique(),
            'pred_score_median': float(np.median(s)),
            'pred_score_top3_mean': topk_mean(s, 3),
            'frac_prn_above_thr': float(np.mean(s > thr_prn)),
        })
    receiver_scores = pd.DataFrame(receiver_scores)
    receiver_scores.to_csv(OUT_DIR / 'validation_receiver_scores.csv', index=False)
    display(prn_val_scores.head())
    display(receiver_scores.head())
else:
    print('No trained torch model; using baseline score tables if available.')

## 12. v2 설계 — Masked PRN reconstruction

다음 단계에서는 같은 시간창의 일부 PRN node를 mask하고, 나머지 PRN과 과거 맥락으로 복원합니다.

학습 objective:

```text
L = L_next_prediction + λ L_masked_prn_reconstruction
```

의미:

- next prediction: 각 PRN tracking 상태의 정상 시간 진화를 학습
- masked reconstruction: 정상 상태에서 다른 PRN들과의 관계로 특정 PRN의 정상 feature를 어느 정도 설명할 수 있는지 학습

주의:

- PRN별 raw peak shape가 서로 같다고 가정하지 않습니다.
- 모델은 PRN ID embedding, 현재 보이는 PRN 집합, feature dynamics를 통해 정상 관계를 학습합니다.
- geometry-aware edge를 추가하면 elevation/azimuth/LOS similarity 기반 관계까지 들어갑니다.

In [ ]:
# v2/v3 구현 메모: 다음 셀은 설계 skeleton입니다.

class_design = {
    'node': 'tracked PRN in a receiver window',
    'node_features': feature_cols,
    'edge_v1': 'fully-connected among tracked PRNs',
    'edge_v2': ['abs(elevation_i-elevation_j)', 'angular_separation', 'abs(doppler_i-doppler_j)', 'LOS dot product'],
    'objectives': ['next-window prediction', 'masked-PRN reconstruction'],
    'anomaly_scores': ['prediction error', 'reconstruction error', 'cross-PRN common-mode error'],
}
print(json.dumps(class_design, indent=2, ensure_ascii=False)[:4000])

## 13. 산출물

이 Notebook이 생성하는 기본 산출물:

```text
artifacts/normal_only_prn_graph_training/
├── gru_next_window_predictor.pt
├── training_history.csv
└── validation_receiver_scores.csv
```

다음으로 연결할 작업:

1. 실제 normal GNSS-SDR tracking dump 경로를 `CFG.TRACKING_FILES`에 넣기
2. 정상 run/day/location 단위 split으로 변경
3. TEXBAT ds1~ds8을 GNSS-SDR로 처리한 tracking dump를 같은 feature pipeline에 통과
4. `prediction error + reconstruction error + cross-PRN common-mode score`로 external test 평가